
# Analyse complète et démonstration du détecteur d’inspection

Ce notebook est **structuré et pédagogique**.  
Il présente **clairement**, dans l’ordre :

1. Exploration du jeu de données  
2. Paramètres **choisis** (sens direct)  
3. Paramètres **estimés** (sens inverse)  
4. Comparaison choisi vs estimé  
5. Construction du détecteur (`y_true`, `y_score`)  
6. Précision / rappel  
7. Courbe ROC  
8. Analyse du coût  

Objectif : **comprendre et démontrer**, pas seulement calculer.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 1. Exploration initiale des données

In [ ]:

# Chargement des données
data = pd.read_csv("sample_system_dataset_0.csv")

# Suppression de la colonne inutile
data = data.drop(columns=["Unnamed: 0"])

# Aperçu
display(data.head())

# Types des colonnes
print("Types des colonnes :")
print(data.dtypes)

# Statistiques descriptives
display(data.describe())


## 2. Sens direct : paramètres choisis

In [ ]:

# Paramètres fixés volontairement
p_true = 0.6          # usure réelle
kappa_true = 20       # qualité de l'inspection

# Paramètres Beta correspondants
a_true = kappa_true * p_true
b_true = kappa_true * (1 - p_true)

params_choisis = pd.DataFrame({
    "Paramètre": ["p", "kappa", "a", "b"],
    "Signification": [
        "usure réelle supposée",
        "qualité de l’inspection",
        "paramètre Beta lié à p",
        "paramètre Beta lié à 1-p"
    ],
    "Valeur": [p_true, kappa_true, a_true, b_true]
})

params_choisis


## 3. Génération des observations du détecteur W

In [ ]:

# Simulation des sorties du détecteur (pour la démonstration)
rng = np.random.default_rng(0)
W = rng.beta(a_true, b_true, size=5000)

pd.DataFrame(W, columns=["W"]).describe()


## 4. Sens inverse : paramètres estimés à partir de W

In [ ]:

# Estimation par méthode des moments
m = W.mean()
v = W.var(ddof=0)

kappa_hat = m * (1 - m) / v - 1
a_hat = kappa_hat * m
b_hat = kappa_hat * (1 - m)
p_hat = a_hat / (a_hat + b_hat)

params_estimes = pd.DataFrame({
    "Paramètre": ["p_hat", "kappa_hat", "a_hat", "b_hat"],
    "Signification": [
        "usure estimée",
        "qualité estimée",
        "paramètre Beta estimé",
        "paramètre Beta estimé"
    ],
    "Valeur": [p_hat, kappa_hat, a_hat, b_hat]
})

params_estimes


## 5. Comparaison : sens direct vs sens inverse

In [ ]:

pd.DataFrame({
    "Paramètre": ["p", "kappa", "a", "b"],
    "Choisi": [p_true, kappa_true, a_true, b_true],
    "Estimé": [p_hat, kappa_hat, a_hat, b_hat]
})


## 6. Construction du détecteur sur les données réelles

In [ ]:

# Création d'une colonne datetime
data["event_datetime"] = pd.to_datetime(
    data["event_date"] + " " + data["event_time"]
)

# Tri chronologique
data = data.sort_values(
    ["system_id", "component_id", "event_datetime"]
)

# Sélection des inspections
inspections = data[data["event_type"] == "inspection"].copy()


### 6.1 Vérité terrain y_true

In [ ]:

def panne_imminente(row, df):
    future = df[
        (df["system_id"] == row["system_id"]) &
        (df["component_id"] == row["component_id"]) &
        (df["event_datetime"] > row["event_datetime"])
    ]
    next_failure = future[future["event_type"] == "failure"]
    next_inspection = future[future["event_type"] == "inspection"]

    if next_failure.empty:
        return 0
    if next_inspection.empty:
        return 1

    return int(
        next_failure.iloc[0]["event_datetime"] <
        next_inspection.iloc[0]["event_datetime"]
    )

inspections["y_true"] = inspections.apply(
    lambda r: panne_imminente(r, data), axis=1
)

inspections[["y_true"]].head()


### 6.2 Score du détecteur y_score

In [ ]:

# Temps depuis la dernière inspection
inspections["prev_event_time"] = inspections.groupby(
    ["system_id", "component_id"]
)["event_datetime"].shift(1)

inspections["delta_t"] = (
    inspections["event_datetime"] - inspections["prev_event_time"]
).dt.total_seconds().fillna(0)

# Normalisation -> score continu
inspections["y_score"] = (
    inspections["delta_t"] - inspections["delta_t"].min()
) / (
    inspections["delta_t"].max() - inspections["delta_t"].min()
)

inspections[["delta_t", "y_score"]].head()


## 7. Précision et rappel (pour un seuil donné)

In [ ]:

threshold = 0.5
inspections["y_pred"] = (inspections["y_score"] >= threshold).astype(int)

TP = ((inspections["y_pred"]==1) & (inspections["y_true"]==1)).sum()
FP = ((inspections["y_pred"]==1) & (inspections["y_true"]==0)).sum()
FN = ((inspections["y_pred"]==0) & (inspections["y_true"]==1)).sum()

precision = TP / (TP + FP) if (TP+FP)>0 else 0
recall = TP / (TP + FN) if (TP+FN)>0 else 0

precision, recall


## 8. Courbe ROC

In [ ]:

def roc_points(y_true, y_score, thresholds):
    rows = []
    for s in thresholds:
        y_pred = (y_score >= s).astype(int)
        TP = ((y_pred==1) & (y_true==1)).sum()
        FP = ((y_pred==1) & (y_true==0)).sum()
        FN = ((y_pred==0) & (y_true==1)).sum()
        TN = ((y_pred==0) & (y_true==0)).sum()

        TPR = TP/(TP+FN) if (TP+FN)>0 else 0
        FPR = FP/(FP+TN) if (FP+TN)>0 else 0

        rows.append((s, FPR, TPR))

    return pd.DataFrame(rows, columns=["threshold","FPR","TPR"])

thresholds = np.linspace(0,1,40)
roc_df = roc_points(inspections["y_true"], inspections["y_score"], thresholds)

plt.figure(figsize=(5,5))
plt.plot(roc_df["FPR"], roc_df["TPR"], marker="o")
plt.plot([0,1],[0,1],"--",color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR (rappel)")
plt.title("Courbe ROC du détecteur")
plt.grid(True)
plt.show()


## 9. Analyse du coût en fonction du seuil

In [ ]:

C_FP = 1    # remplacement inutile
C_FN = 10   # panne non anticipée

costs = []
for s in thresholds:
    y_pred = (inspections["y_score"] >= s).astype(int)
    FP = ((y_pred==1) & (inspections["y_true"]==0)).sum()
    FN = ((y_pred==0) & (inspections["y_true"]==1)).sum()
    costs.append(C_FP*FP + C_FN*FN)

cost_df = pd.DataFrame({
    "threshold": thresholds,
    "cost": costs
})

plt.figure()
plt.plot(cost_df["threshold"], cost_df["cost"], marker="o")
plt.xlabel("Seuil")
plt.ylabel("Coût total")
plt.title("Coût total en fonction du seuil")
plt.grid(True)
plt.show()
